In [1]:
import os
import sys
from pathlib import Path
import jax.numpy as np
import matplotlib.pyplot as plt

from gwfast.gwfastGlobals import detectors as det_dict, detPath, MRSUN_SI, MTSUN_SI, uGpc
import gwfast.waveforms as waveforms
from gwfast.detector import Detector
from gwfast.signals import BasicGWSignal, AGNLensedGWSignal
import gwfast.network as network
import gwfast.lensing_utils as lutils
from gwfast.lensing_utils_alt import get_agn_lensed_parameters, PML_time_delay_magnification, dLGridGlob, zGridGlob, einstein_angle

LSC Algorithm Library (LAL) is not installed, only the GWFAST waveform models are available, namely: TaylorF2, IMRPhenomD, IMRPhenomD_NRTidalv2, IMRPhenomHM and IMRPhenomNSBH
TEOBResumS is not installed, only the GWFAST waveform models are available, namely: TaylorF2, IMRPhenomD, IMRPhenomD_NRTidalv2, IMRPhenomHM and IMRPhenomNSBH


In [2]:
# Set up detectors
H1 = Detector('H1', **det_dict['H1'],
              noise_curve_path=Path(detPath)/'observing_scenarios_paper/AplusDesign.txt')

wf_model = waveforms.IMRPhenomHM()

H1_AGN = AGNLensedGWSignal(wf_model=wf_model, detector=H1, fmin=10)
# L1_AGN = AGNLensedGWSignal(wf_model=wf_model, detector=L1, fmin=10)
# V1_AGN = AGNLensedGWSignal(wf_model=wf_model, detector=V1, fmin=10)
# HLV_AGN = network.DetNet({'H1': H1_AGN, 'L1': L1_AGN, 'V1': V1_AGN})

H1_BBH = BasicGWSignal(wf_model=wf_model, detector=H1, fmin=10)
# L1_BBH = BasicGWSignal(wf_model=wf_model, detector=L1, fmin=10)
# V1_BBH = BasicGWSignal(wf_model=wf_model, detector=V1, fmin=10)
# HLV_BBH = network.DetNet({'H1': H1_BBH, 'L1': L1_BBH, 'V1': V1_BBH})

Initializing jax...
Jax local device count: 1
Jax device count: 1
Initializing jax...
Jax local device count: 1
Jax device count: 1


In [3]:
# Define a benchmark event
n = 2
shape = (n,n)
events = {
    'Mc':np.full(shape, 30), 'eta':np.full(shape, 0.24), 'dL':np.full(shape, 0.8), 'theta':np.full(shape, 1.87), 'phi':np.full(shape, 2.66), 
    'iota':np.full(shape, 0.99*np.pi/2), 'psi':np.full(shape, 4), 'tcoal':np.full(shape, 0), 'phase':np.full(shape, 2), 
    'chi1z':np.full(shape, 0.3), 'chi2z':np.full(shape, 0.5), 'chi1x':np.full(shape, 0), 'chi2x':np.full(shape, 0), 'chi1y':np.full(shape, 0), 'chi2y':np.full(shape, 0), 
    'LambdaTilde':np.full(shape, 0), 'deltaLambda':np.full(shape, 0), 'ecc':np.full(shape, 0), 
    'R_orbit':np.full(shape, 20), 'M_lz':np.full(shape, 1e4), 'src_pos':np.full(shape, 0.5)
}
events = {key: val.astype(np.float64) for key, val in events.items()}

# Waveform metadata
sampling_frequency = 1024.  # Hz
minimum_frequency = 10.  # Hz
maximum_frequency = sampling_frequency / 2
N_freq_bins = 1000

# Make frequency grid
minimum_frequency_array = np.full(shape, minimum_frequency)
maximum_frequency_array = np.full(shape, maximum_frequency)
freq_grid = np.geomspace(minimum_frequency_array, maximum_frequency_array, N_freq_bins)
freq_array = np.geomspace(minimum_frequency, maximum_frequency, N_freq_bins)

# Generate strains from H1
FD_strain_unlensed = H1_BBH.GWstrain(freq_grid, events)
FD_strain_lensed = H1_AGN.GWstrain(freq_grid, events)

ValueError: Incompatible shapes for broadcasting: shapes=[(1000, 2, 6, 2), (2, 2)]

In [ ]:
scale = 0.7
fig, axes = plt.subplots(1, 2, figsize=(16*scale, 4*scale),
                        width_ratios=(1, 3), constrained_layout=True)

ax = axes[0]
ax.loglog(freq_array, np.abs(FD_strain_unlensed[:, 0]), 
          alpha=0.7, label='unlensed')
ax.loglog(freq_array, np.abs(FD_strain_lensed[:, 0]), 
          label='lensed', zorder=1)
ax.set_title('Frequency domain strain')
ax.set_xlabel(r'Frequency, $f\,/\,{\rm Hz}$')
ax.set_ylabel(r'$\tilde{h}(f)\,/\,{\rm s}^2$')
ax.set_xlim(9, 550)

# ax = axes[1]
# ax.plot(time_array, TD_strain_unlensed, 
#         alpha=0.7, label='unlensed')
# ax.plot(time_array, TD_strain_lensed, 
#         label='lensed', zorder=1)
# ax.set_title('Time domain strain')
# ax.set_xlabel(r'Time, $t\,/\,{\rm s}$')
# ax.set_ylabel(r'$h(t)$')
# ax.set_xlim(0.0, 8.0)

for ax in axes:
    ax.legend()